# Django, URL Routing

## Introduction to URL Routing
---

In this lesson, you'll learn how **URL routing** works in Django.

URL routing is the mechanism that maps URL addresses to the code that handles them - your views.

1. [URLconf Basics](#URLconf-Basics),
    - [What is URLconf](#What-is-URLconf),
    - [The path() Function](#The-path()-Function),
2. [URL Parameters](#URL-Parameters),
    - [Path Converters](#Path-Converters),
    - [Capturing Values from URLs](#Capturing-Values-from-URLs),
3. [Modular URLs with include()](#Modular-URLs-with-include()),
    - [App-Level URL Configuration](#App-Level-URL-Configuration),
    - [URL Prefixes](#URL-Prefixes),
4. [URL Naming and reverse()](#URL-Naming-and-reverse()),
    - [Why Name URLs](#Why-Name-URLs),
    - [Using reverse() in Code](#Using-reverse()-in-Code),
    - [URL Namespaces](#URL-Namespaces),
5. [🧠 EXERCISE 🧠, Create URL Patterns](#🧠-EXERCISE-🧠,-Create-URL-Patterns).

<br>

## URLconf Basics

---

### What is URLconf

---

A **URLconf** (URL configuration) is a mapping between URL patterns and view functions.

Think of it as a **table of contents** for your website - it tells Django which code to run for each URL.

<br>

| URL | View Function | What it does |
|-----|---------------|-------------|
| `/` | `home` | Shows homepage |
| `/books/` | `book_list` | Lists all books |
| `/books/5/` | `book_detail` | Shows book with ID 5 |
| `/about/` | `about` | Shows about page |

<br>

The URLconf is defined in `urls.py` files using a list called `urlpatterns`.

<br>

### The path() Function

---

The `path()` function is the primary way to define URL patterns in Django.

<br>

**Syntax:**

```python
path(route, view, kwargs=None, name=None)
```

| Parameter | Description |
|-----------|-------------|
| `route` | URL pattern to match (e.g., `'books/'`) |
| `view` | View function or class to call |
| `kwargs` | Additional arguments to pass to the view |
| `name` | Name for the URL pattern (for reverse lookups) |

In [ ]:
# Example: Basic urls.py structure

from django.urls import path
from . import views  # Import views from current app

urlpatterns = [
    path('', views.home, name='home'),           # Matches: /
    path('books/', views.book_list, name='book_list'),  # Matches: /books/
    path('about/', views.about, name='about'),   # Matches: /about/
]

<br>

**Key points about URL routes:**

- Routes **don't start with a slash** - Django adds it automatically
- Empty string `''` matches the root URL `/`
- Django tries patterns **in order** - first match wins
- Trailing slashes are **recommended** for consistency

In [ ]:
# Example: Simple view functions for the URLs above

from django.http import HttpResponse

def home(request):
    return HttpResponse("Welcome to the Bookstore!")

def book_list(request):
    return HttpResponse("Here's a list of all books.")

def about(request):
    return HttpResponse("About our bookstore.")

<br>

## URL Parameters

---

### Path Converters

---

Often you need **dynamic URLs** that capture values - like a book ID or a username.

Django uses **path converters** to capture parts of the URL and pass them to your view.

<br>

**Syntax:** `<converter:name>`

| Converter | Matches | Example |
|-----------|---------|--------|
| `int` | Positive integers | `<int:pk>` → `1`, `42`, `999` |
| `str` | Non-empty string (excludes `/`) | `<str:title>` → `hello`, `my-book` |
| `slug` | ASCII letters, numbers, hyphens, underscores | `<slug:slug>` → `my-first-post` |
| `uuid` | UUID format | `<uuid:id>` → `075194d3-...` |
| `path` | Any string including `/` | `<path:file>` → `docs/intro.txt` |

In [ ]:
# Example: URL patterns with parameters

from django.urls import path
from . import views

urlpatterns = [
    # Integer parameter - for database IDs
    path('books/<int:pk>/', views.book_detail, name='book_detail'),
    # Matches: /books/1/, /books/42/, /books/999/
    
    # Slug parameter - for SEO-friendly URLs
    path('books/<slug:slug>/', views.book_by_slug, name='book_by_slug'),
    # Matches: /books/django-for-beginners/, /books/python-crash-course/
    
    # Multiple parameters
    path('books/<int:year>/<int:month>/', views.books_archive, name='books_archive'),
    # Matches: /books/2024/03/, /books/2023/12/
]

<br>

### Capturing Values from URLs

---

When a URL matches, the captured values are passed to your view as **keyword arguments**.

In [ ]:
# Example: View functions receiving URL parameters

from django.http import HttpResponse

def book_detail(request, pk: int):
    """Display a single book by its primary key."""
    # pk is automatically converted to int by Django
    return HttpResponse(f"Showing book with ID: {pk}")

def book_by_slug(request, slug: str):
    """Display a book by its URL slug."""
    return HttpResponse(f"Showing book: {slug}")

def books_archive(request, year: int, month: int):
    """Display books published in a specific month."""
    return HttpResponse(f"Books from {month}/{year}")

<br>

**Type conversion happens automatically:**

| URL | Converter | Received Value |
|-----|-----------|----------------|
| `/books/42/` | `<int:pk>` | `pk=42` (integer) |
| `/books/hello/` | `<str:title>` | `title="hello"` (string) |
| `/books/my-book/` | `<slug:slug>` | `slug="my-book"` (string) |

In [ ]:
# Example: Real-world book detail view

from django.http import HttpResponse, Http404

# Simulated database of books
BOOKS = {
    1: {'title': 'Django for Beginners', 'author': 'William Vincent'},
    2: {'title': 'Two Scoops of Django', 'author': 'Daniel Feldroy'},
    3: {'title': 'Django for Professionals', 'author': 'William Vincent'},
}

def book_detail(request, pk: int):
    """Display book details or return 404."""
    book = BOOKS.get(pk)
    if book is None:
        raise Http404(f"Book with ID {pk} not found")
    return HttpResponse(f"<h1>{book['title']}</h1><p>By {book['author']}</p>")

# URL: /books/1/  →  "Django for Beginners by William Vincent"
# URL: /books/99/ →  404 Not Found

<br>

## Modular URLs with include()

---

### App-Level URL Configuration

---

As your project grows, putting all URLs in one file becomes messy. Django's `include()` function lets you **split URLs by app**.

<br>

**The pattern:**

1. Create `urls.py` in each app
2. Use `include()` in the project's `urls.py` to reference them

In [ ]:
# catalog/urls.py - App-specific URLs

from django.urls import path
from . import views

app_name = 'catalog'  # Namespace for this app's URLs

urlpatterns = [
    path('', views.book_list, name='book_list'),
    path('<int:pk>/', views.book_detail, name='book_detail'),
    path('<int:pk>/reviews/', views.book_reviews, name='book_reviews'),
]

In [ ]:
# bookstore_project/urls.py - Project-level URLs

from django.contrib import admin
from django.urls import path, include

urlpatterns = [
    path('admin/', admin.site.urls),
    path('books/', include('catalog.urls')),    # All /books/... URLs
    path('accounts/', include('accounts.urls')), # All /accounts/... URLs
]

<br>

**Result:**

| Full URL | Handled by |
|----------|------------|
| `/books/` | `catalog.views.book_list` |
| `/books/5/` | `catalog.views.book_detail` |
| `/books/5/reviews/` | `catalog.views.book_reviews` |
| `/accounts/login/` | `accounts.views.login` |

<br>

### URL Prefixes

---

The first argument to `path()` before `include()` becomes a **prefix** for all included URLs.

<br>

In [ ]:
# Example: Different prefixes for the same app URLs

from django.urls import path, include

urlpatterns = [
    # English site
    path('en/books/', include('catalog.urls')),
    # Matches: /en/books/, /en/books/5/, etc.
    
    # German site
    path('de/buecher/', include('catalog.urls')),
    # Matches: /de/buecher/, /de/buecher/5/, etc.
    
    # API version
    path('api/v1/', include('catalog.urls')),
    # Matches: /api/v1/, /api/v1/5/, etc.
]

<br>

**Best practice:** Keep your project's `urls.py` clean - only use `include()` to delegate to apps:

```python
# GOOD: Project urls.py with includes only
urlpatterns = [
    path('admin/', admin.site.urls),
    path('', include('pages.urls')),
    path('books/', include('catalog.urls')),
    path('accounts/', include('accounts.urls')),
]

# AVOID: Mixing includes with direct views
urlpatterns = [
    path('admin/', admin.site.urls),
    path('', views.home),  # Direct view in project urls
    path('books/', include('catalog.urls')),
    path('contact/', views.contact),  # Another direct view
]
```

<br>

## URL Naming and reverse()

---

### Why Name URLs

---

Hardcoding URLs in your code is **fragile**. If you change a URL pattern, you have to update it everywhere.

<br>

**The problem:**

In [ ]:
# BAD: Hardcoded URLs

def some_view(request):
    # If URL changes from /books/ to /catalog/, this breaks!
    return redirect('/books/5/')  

# In template (also BAD):
# <a href="/books/5/">View book</a>

<br>

**The solution:** Name your URLs and use `reverse()` to generate them.

In [ ]:
# urls.py - Name your URLs

from django.urls import path
from . import views

urlpatterns = [
    path('', views.book_list, name='book_list'),
    path('<int:pk>/', views.book_detail, name='book_detail'),
    #                                    ^^^^^^^^^^^^^^^^^
    #                                    URL name for reverse lookup
]

<br>

### Using reverse() in Code

---

The `reverse()` function generates a URL from its name.

In [ ]:
# GOOD: Using reverse() in views

from django.urls import reverse
from django.shortcuts import redirect

def some_view(request):
    # Generate URL dynamically - if URL pattern changes, this still works!
    url = reverse('book_detail', kwargs={'pk': 5})
    # url = '/books/5/'
    return redirect(url)

# Even simpler with redirect shortcut:
def another_view(request):
    return redirect('book_detail', pk=5)

In [ ]:
# Example: reverse() with different URL patterns

from django.urls import reverse

# Simple URL (no parameters)
url1 = reverse('book_list')
# Result: '/books/'

# URL with one parameter
url2 = reverse('book_detail', kwargs={'pk': 42})
# Result: '/books/42/'

# Alternative syntax with args (positional)
url3 = reverse('book_detail', args=[42])
# Result: '/books/42/'

# URL with multiple parameters
url4 = reverse('books_archive', kwargs={'year': 2024, 'month': 3})
# Result: '/books/2024/03/'

<br>

**In templates**, use the `{% url %}` tag instead of `reverse()`:

In [ ]:
# Template examples (not actual Python, just for illustration)

template_examples = """
<!-- Simple URL -->
<a href="{% url 'book_list' %}">All Books</a>

<!-- URL with parameter -->
<a href="{% url 'book_detail' pk=book.pk %}">{{ book.title }}</a>

<!-- URL with multiple parameters -->
<a href="{% url 'books_archive' year=2024 month=3 %}">March 2024</a>
"""
print(template_examples)

<br>

### URL Namespaces

---

When you have multiple apps with similar URL names (like `detail` or `list`), **namespaces** prevent conflicts.

<br>

**Define a namespace** with `app_name` in your app's `urls.py`:

In [ ]:
# catalog/urls.py

from django.urls import path
from . import views

app_name = 'catalog'  # This is the namespace

urlpatterns = [
    path('', views.book_list, name='list'),
    path('<int:pk>/', views.book_detail, name='detail'),
]

In [ ]:
# accounts/urls.py

from django.urls import path
from . import views

app_name = 'accounts'  # Different namespace

urlpatterns = [
    path('', views.user_list, name='list'),      # Same name, different namespace
    path('<int:pk>/', views.user_detail, name='detail'),
]

In [ ]:
# Using namespaced URLs

from django.urls import reverse

# Use namespace:name syntax
catalog_url = reverse('catalog:detail', kwargs={'pk': 5})
# Result: '/books/5/'

accounts_url = reverse('accounts:detail', kwargs={'pk': 5})
# Result: '/accounts/5/'

In [ ]:
# Template with namespaced URLs

template_namespace_example = """
<!-- Catalog book detail -->
<a href="{% url 'catalog:detail' pk=book.pk %}">View Book</a>

<!-- Accounts user detail -->
<a href="{% url 'accounts:detail' pk=user.pk %}">View Profile</a>
"""
print(template_namespace_example)

<br>

## Common URL Patterns

---

Here are some common URL patterns you'll encounter:

In [ ]:
# Example: Common CRUD URL patterns

from django.urls import path
from . import views

app_name = 'catalog'

urlpatterns = [
    # List all books
    path('', views.book_list, name='book_list'),
    # GET /books/
    
    # Create a new book
    path('new/', views.book_create, name='book_create'),
    # GET /books/new/ (show form)
    # POST /books/new/ (create book)
    
    # View a single book
    path('<int:pk>/', views.book_detail, name='book_detail'),
    # GET /books/5/
    
    # Edit a book
    path('<int:pk>/edit/', views.book_update, name='book_update'),
    # GET /books/5/edit/ (show form)
    # POST /books/5/edit/ (update book)
    
    # Delete a book
    path('<int:pk>/delete/', views.book_delete, name='book_delete'),
    # GET /books/5/delete/ (confirm)
    # POST /books/5/delete/ (delete)
]

In [ ]:
# Example: Nested resources

from django.urls import path
from . import views

app_name = 'catalog'

urlpatterns = [
    # Books
    path('books/', views.book_list, name='book_list'),
    path('books/<int:pk>/', views.book_detail, name='book_detail'),
    
    # Reviews (nested under books)
    path('books/<int:book_pk>/reviews/', views.review_list, name='review_list'),
    path('books/<int:book_pk>/reviews/<int:pk>/', views.review_detail, name='review_detail'),
    
    # Authors
    path('authors/', views.author_list, name='author_list'),
    path('authors/<int:pk>/', views.author_detail, name='author_detail'),
    path('authors/<int:pk>/books/', views.author_books, name='author_books'),
]

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **URLconf** | `urlpatterns` list in `urls.py` - maps URLs to views |
| **path()** | `path(route, view, name=...)` - defines URL pattern |
| **Converters** | `<int:pk>`, `<str:name>`, `<slug:slug>` - capture URL parts |
| **include()** | Delegate URLs to app-specific `urls.py` |
| **URL names** | Always name URLs for `reverse()` lookups |
| **reverse()** | Generate URLs from names - avoids hardcoding |
| **Namespaces** | `app_name` prevents URL name conflicts |

<br>

### 🧠 EXERCISE 🧠, Create URL Patterns

---

Using the `bookstore` project from previous lessons:

1. Create URL patterns in `catalog/urls.py` for:
   - List all books at `/` (name: `book_list`)
   - Book detail at `/<int:pk>/` (name: `book_detail`)
   - Books by author at `/author/<int:author_id>/` (name: `books_by_author`)
2. Set `app_name = 'catalog'` for namespacing
3. Create simple view functions in `catalog/views.py` that return HttpResponse
4. Include catalog URLs in the project's `urls.py` at `/books/`
5. Test by running the server and visiting the URLs

<details>
    <summary>▶️ Solution</summary>
    
**1-2. Edit `catalog/urls.py`:**

```python
from django.urls import path
from . import views

app_name = 'catalog'

urlpatterns = [
    path('', views.book_list, name='book_list'),
    path('<int:pk>/', views.book_detail, name='book_detail'),
    path('author/<int:author_id>/', views.books_by_author, name='books_by_author'),
]
```

**3. Edit `catalog/views.py`:**

```python
from django.http import HttpResponse

def book_list(request):
    return HttpResponse("List of all books")

def book_detail(request, pk: int):
    return HttpResponse(f"Book detail for ID: {pk}")

def books_by_author(request, author_id: int):
    return HttpResponse(f"Books by author ID: {author_id}")
```

**4. Edit `bookstore_project/urls.py`:**

```python
from django.contrib import admin
from django.urls import path, include

urlpatterns = [
    path('admin/', admin.site.urls),
    path('books/', include('catalog.urls')),
]
```

**5. Test:**

```bash
python manage.py runserver
```

Visit:
- `http://127.0.0.1:8000/books/` → "List of all books"
- `http://127.0.0.1:8000/books/5/` → "Book detail for ID: 5"
- `http://127.0.0.1:8000/books/author/3/` → "Books by author ID: 3"
</details>

<br>

### 🧠 EXERCISE 🧠, Practice reverse()

---

Practice using `reverse()` to generate URLs:

1. Open the Django shell: `python manage.py shell`
2. Import `reverse`: `from django.urls import reverse`
3. Generate URLs for:
   - The book list
   - Book detail for ID 42
   - Books by author with ID 7
4. Create a view that redirects from `/` to the book list

<details>
    <summary>▶️ Solution</summary>
    
**1-3. In Django shell:**

```python
>>> from django.urls import reverse

>>> reverse('catalog:book_list')
'/books/'

>>> reverse('catalog:book_detail', kwargs={'pk': 42})
'/books/42/'

>>> reverse('catalog:books_by_author', kwargs={'author_id': 7})
'/books/author/7/'

>>> exit()
```

**4. Add redirect view to `catalog/views.py`:**

```python
from django.shortcuts import redirect

def home(request):
    return redirect('catalog:book_list')
```

**Add to `bookstore_project/urls.py`:**

```python
from catalog.views import home

urlpatterns = [
    path('admin/', admin.site.urls),
    path('', home, name='home'),  # Redirect to book list
    path('books/', include('catalog.urls')),
]
```

Now visiting `http://127.0.0.1:8000/` redirects to `/books/`.
</details>

---